# E- handelsanalyysis

## Project syfte

Analysera försäljningsdata for att idetentifera använbara mönster och ge ledgning beslut innan nästa kampanjperiod.

## Frågeställningar

1. Vilka produkt kategorier driver mest intäkt?
2. Vilka städer står för störst intäkt?
3. Hur ser ut Försaljningen över tid?
4. Hur ser en typisk order ut? – AOV (Average Order Value) och spridning.
5. Topp-listor – topp-3 kategorier efter intäkt.
6. Finns det avvikelser eller något oväntat mönster i data?



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
from src.io_utils import load_data

df = load_data()

print("Data loaded")


In [ ]:
df.head()

In [ ]:
print("Ecommerce data Information")
df.info()


In [ ]:
print("Statistics")

print(df.describe())
print()

In [ ]:

df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Date:" "From", df["date"] . min() , "to" ,df["date"].max())

In [48]:
print("Cities:" ,list(df["city"].unique()))
print()

print("Produkt Categories:" ,list(df["category"].unique()))
print()

Cities: ['Stockholm', 'Malmö', 'Uppsala', 'Göteborg', 'Västerås']

Produkt Categories: ['Sports', 'Clothing', 'Home', 'Electronics', 'Toys', 'Beauty']



## Beräkna total Intäkten

In [ ]:
#Total intäkt och Antal enheter

total_intäkt = df["revenue"].sum()
print(f"Total intäkt: {total_intäkt:,.2f} kr")
print()

antal_enheter = df["units"].sum()
print(f"Total antal enheter: {antal_enheter}")


## Intäkt per stad

In [ ]:
#Intäkt Per Stad

#Tabell
import importlib
import src.metrics as metrics

stads_intäkt = metrics.intäkt_per_stad(df)

stads_intäkt_tabell = stads_intäkt.reset_index()
stads_intäkt_tabell.columns = ["Stad" , "Total intäkt"]

display(stads_intäkt_tabell)



In [ ]:
högst_intäkt_stad = stads_intäkt.idxmax()
hogsta_stads_intakt = stads_intäkt.max()

print(f"Stad med högst intäkt: {högst_intäkt_stad}")
print(f"Högsta stads intäkten: {hogsta_stads_intakt:,.2f} kr")

In [ ]:
# kategori intäkt 

In [ ]:
# Top tre kategori

In [ ]:
# månads intäkt

In [ ]:
#aov

In [ ]:
#ordervalue distribution


import importlib
import src.metrics as metrics

importlib.reload(metrics)

order_values = metrics.ordervarden(df)

print("Antal unika orderr:", order_values.size)
print(f"Lägsta ordervärde: {order_values.min():,.2f} kr")
print(f"Median för ordervärde: {order_values.median():,.2f} kr")
print(f"Högsta ordervärde: {order_values.max():,.2f} kr")

In [ ]:
nyckeltal_tabell = pd.DataFrame({
    "Nyckeltal": [
        "Total intäkt",
        "Totalt antal sålda enheter",
        "Antal unika ordrar",
        "Stad med högst intäkt",
    ],
    "Resultat": [
        f"{total_intäkt:,.2f} kr",
        f"{antal_enheter:,}",
        f"{order_values.size:,}",
        str(stads_intäkt.idxmax()),
        
    ]
})

display(nyckeltal_tabell)


# Visualization

In [ ]:
#plot intäkt per stad

import importlib
import src.viz as viz

importlib.reload(viz)

viz.plot_intakt_per_stad(stads_intäkt)

In [ ]:
#plot kategory

In [ ]:
#plot topp tre kategori

In [ ]:
#plot försäjlning over tid

In [ ]:
#Aov

In [ ]:
#order value distributin


import importlib
import src.viz as viz

importlib.reload(viz)

viz.plot_ordervarden(order_values)

## Analysera  avvikelser och något oväntat mönster


In [ ]:
# Saknande värden per kolumn
print(df.isnull().sum())

# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Invalid or unusual values
print("Price <= 0:", (df["price"] <= 0).sum())
print("Units <= 0:", (df["units"] <= 0).sum())
print("Revenue < 0:", (df["revenue"] < 0).sum())

In [52]:

import numpy as np
import pandas as pd

kolumner = ["price", "units", "revenue"]

sammanfattningar = []
outlier_limit = {}
outlier_data = {}

for kolumn in kolumner:
    values = df[kolumn].dropna().to_numpy()

    minimum = np.min(values)
    q1 = np.percentile(values, 25)
    median = np.median(values)
    q3 = np.percentile(values, 75)
    maximum = np.max(values)

    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    # Identifiera potentiella avvikande värden
    mask = (df[kolumn] < lower_limit) | (
        df[kolumn] > upper_limit
    )
    avvikelser = df.loc[mask]

    sammanfattningar.append({
        "Kolumn": kolumn,
        "Minimum": minimum,
        "Q1": q1,
        "Median": median,
        "Q3": q3,
        "Maximum": maximum,
        "IQR": iqr,
        "Antal avvikelser": len(avvikelser)
    })

    outlier_limit[kolumn] = (
        lower_limit, upper_limit
    )
    outlier_data[kolumn] = avvikelser

sammanfattning = pd.DataFrame(sammanfattningar)

display(sammanfattning.round(2))


,Kolumn,Minimum,Q1,Median,Q3,Maximum,IQR,Antal avvikelser
0,price,50.0,254.96,413.04,624.43,2092.53,369.47,88
1,units,1.0,2.00,3.00,4.00,9.00,2.00,7
2,revenue,50.0,577.96,1088.69,1884.75,9192.60,1306.79,111


In [53]:
alla_avvikelser = pd.concat(
    [outlier_data[kolumn] for kolumn in kolumner]
).drop_duplicates()

print(f"Totalt antal unika rader med avvikelser: {len(alla_avvikelser)}")

display(
    alla_avvikelser[
        ["order_id", "date", "city",
         "category", "price", "units", "revenue"]
    ].sort_values("revenue", ascending=False)
)


Totalt antal unika rader med avvikelser: 162


,order_id,date,city,category,price,units,revenue
1411,101411,2024-02-26,Stockholm,Electronics,1838.52,5,9192.60
1339,101339,2024-02-26,Göteborg,Electronics,1703.37,5,8516.85
1491,101491,2024-06-04,Västerås,Electronics,2092.53,4,8370.12
1733,101733,2024-06-05,Uppsala,Electronics,1227.73,6,7366.38
1518,101518,2024-01-12,Göteborg,Electronics,1403.69,5,7018.45
...,...,...,...,...,...,...,...
1533,101533,2024-05-10,Västerås,Electronics,1264.50,1,1264.50
223,100223,2024-04-09,Malmö,Electronics,1257.95,1,1257.95
1408,101408,2024-04-20,Uppsala,Electronics,1236.04,1,1236.04
1722,101722,2024-06-27,Stockholm,Electronics,1222.72,1,1222.72
